In [19]:
import torch
from torchinfo import summary
from torch.utils.data import DataLoader, random_split
import pandas as pd
import os
from dotenv import load_dotenv
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
import sys
from tqdm.notebook import tqdm
import pickle

from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, f1_score, classification_report

from src.transformer.transformer_architecture import KeypointTransformer, LitKeypointTransformer
from src.image_processing.mediapipe import MediaPipeProcessor
from src.datasets.ipn_data import IPNData

In [2]:
load_dotenv()
GESTURE_VIDEOS_PATH = os.getenv("IPN_GESTURE_VIDEOS")

In [3]:
# Dataset
df = pd.read_csv(f"{GESTURE_VIDEOS_PATH}/metadata.csv")
df.head()

,Video Name,Frames,Sex,Hand,Background,Illumination,People in Scene,Background Motion,Set,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,1CM1_4_R__229,3751,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1CM1_4_R__230,3684,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1CM1_4_R__231,3747,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1CM1_4_R__232,3858,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1CM42_11_R__205,3686,M,Right,Plain,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
ds_raw = IPNData()
mpp = MediaPipeProcessor()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774300603.445703   18615 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774300603.454338   18620 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [5]:
label_id_dict = ds_raw.get_label_id_dict()
num_videos = ds_raw.get_num_videos()
video_info_dict = ds_raw.get_video_info()

In [6]:
print(len(label_id_dict))
print(len(video_info_dict))

14
5649


In [8]:
label_to_index = {}

for k in label_id_dict:
    label, gesture = label_id_dict[k]
    label_to_index[label] = int(k)

In [ ]:
N_V = num_videos
ds = []

for i in tqdm(range(N_V)):
    for j in range(sys.maxsize):
        try:
            frames, label, ID = ds_raw.__getitem__(i, j)

            keypoint_sequence = mpp.process_video(frames)

            ds.append([keypoint_sequence, label_to_index[label]]) # using ID as label
        except:
            break

In [9]:
#Saving new dataset

with open("data/dataset.pkl", "wb") as f:
    pickle.dump(ds, f)

print("Saved!")

Saved!


In [9]:
with open("data/dataset.pkl", "rb") as f:
    ds = pickle.load(f)

In [10]:
N_F = len(video_info_dict)
TEST_F = int(0.1*N_F)
TRAIN_VAL_F = N_F - TEST_F

train_size = int(0.9 * TRAIN_VAL_F)
val_size = TRAIN_VAL_F - train_size

test_ds = ds[TRAIN_VAL_F:]
train_val_ds = ds[:TRAIN_VAL_F]

train_ds, val_ds = random_split(train_val_ds, [train_size, val_size])

In [11]:
def collate_fn(batch):
    sequences, labels = zip(*batch)

    sequences = [torch.tensor(s, dtype=torch.float32) for s in sequences]

    # Padding
    padded_sequences = pad_sequence(sequences, batch_first=True)

    # Masking
    mask = torch.zeros(size=padded_sequences.shape[:2], dtype=torch.bool)
    for(i, s) in enumerate(sequences):
        mask[i, :len(s)] = True

    # Labeling
    labels = torch.tensor([int(l) for l in labels])

    return padded_sequences, mask, labels

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=32, collate_fn=collate_fn)

input_size = 126
d_model = 132

num_classes = len(label_id_dict)

model = LitKeypointTransformer(input_size=input_size, d_model=d_model, num_classes=num_classes)

early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min")

trainer = L.Trainer(
    max_epochs=60,
    accelerator="auto",
    devices=1,
    callbacks=[early_stopping]
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [13]:
x_batch, mask, y_batch = next(iter(train_loader))
logits = model(x_batch)  
print("x:", x_batch.shape, "y:", y_batch.shape, "logits:", logits.shape)

x: torch.Size([32, 358, 126]) y: torch.Size([32]) logits: torch.Size([32, 14])


In [14]:
trainer.fit(model, train_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 3070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type                | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | model   | KeypointTransformer | 1.3 M  | train | 0    
1 | loss_fn | CrossEntropyLoss    | 0      | train | 0    
----------------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=40` reached.


In [ ]:
from pathlib import Path
save_direct = Path("checkpoints")
trainer.save_checkpoint(save_direct / "keypoint_transformer_extensive.ckpt") 

`weights_only` was not set, defaulting to `False`.


In [16]:
model.eval()

LitKeypointTransformer(
  (model): KeypointTransformer(
    (input_proj): Sequential(
      (0): Linear(in_features=126, out_features=132, bias=True)
    )
    (pos_encode): SinusoidalPositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncodingLayer(
        (attention): MultiHeadAttention(
          (w_q): Linear(in_features=132, out_features=132, bias=True)
          (w_k): Linear(in_features=132, out_features=132, bias=True)
          (w_v): Linear(in_features=132, out_features=132, bias=True)
          (output): Linear(in_features=132, out_features=132, bias=True)
        )
        (norm1): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
        (ffn): Sequential(
          (0): Linear(in_features=132, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_f

In [ ]:
#Testing model

In [17]:
all_labels = []
all_preds = []

for i in tqdm(range(len(test_ds))):    
    keypoint_sequence, label = test_ds[i]

    x = torch.tensor(keypoint_sequence, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, 126)

    # prediction
    with torch.no_grad():
        logits = model(x)
        predicted_idx = torch.argmax(logits, dim=-1).item()

    all_labels.append(label)
    all_preds.append(predicted_idx)


  0%|          | 0/564 [00:00<?, ?it/s]

In [20]:
# F1 score and accuracy (again)
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="macro")
class_report = classification_report(all_labels, all_preds)

print(accuracy)
print(f1)
print(class_report)

0.8421985815602837
0.7485571253830187
              precision    recall  f1-score   support

           0       0.86      0.93      0.89       148
           1       0.95      0.95      0.95       101
           2       0.92      0.95      0.93        98
           3       0.48      0.68      0.57        19
           4       0.58      0.70      0.64        20
           5       0.94      0.85      0.89        20
           6       1.00      0.75      0.86        20
           7       0.75      0.47      0.58        19
           8       0.89      0.85      0.87        20
           9       0.73      0.80      0.76        20
          10       0.86      0.30      0.44        20
          11       0.90      0.45      0.60        20
          12       0.73      0.80      0.76        20
          13       0.64      0.84      0.73        19

    accuracy                           0.84       564
   macro avg       0.80      0.74      0.75       564
weighted avg       0.85      0.84      0.8